# 🛵 Sentinel — Two-Wheeler & Small-Car Specialized Deep Training (Google Colab)
### High-Res (1280px) • P2 Small-Object Detection Head • Dense Traffic Crowd Synthesis

This notebook specifically solves the problem of **missed two-wheelers (scooters, motorcycles)** and **distant small cars** in elevated CCTV footage.

---
### 🎯 Key Enhancements:
1. **1280px Ultra High-Resolution Multi-Scale Training**: Eliminates downsampling loss on tiny 15px-30px distant scooters.
2. **Small-Object Loss Rebalancing**: Increases bounding-box IoU gradient weights (, ) for two-wheelers and compact cars.
3. **Heavy Downscale Jitter ()**: Synthetically shrinks vehicles during training so the AI learns to recognize microscopic bikes and distant hatchbacks.
4. **Dense Intersection Crowd Clustering**: Trains with Mosaic & MixUp to recognize bunched two-wheelers waiting at traffic signals.

In [ ]:
# Optional: Mount Google Drive to Permanently Save Checkpoints
import os
from google.colab import drive
try:
    drive.mount("/content/drive")
    save_dir = "/content/drive/MyDrive/Sentinel_AI_Models"
    os.makedirs(save_dir, exist_ok=True)
    print(f"📁 Google Drive connected! Models will be permanently backed up to: {save_dir}")
except Exception as e:
    print("ℹ️ Continuing with local Colab disk.")

In [ ]:
# Cell 1: Check High-Memory NVIDIA GPU
!nvidia-smi
import torch
print(f"CUDA Ready: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)} ({torch.cuda.get_device_properties(0).total_memory / (1024**3):.2f} GB VRAM)")

In [ ]:
# Cell 2: Install Ultralytics YOLO & Dependencies
!pip install -q ultralytics albumentations pyyaml gdown

In [ ]:
# Cell 3: Unpack Dataset & Auto-Detect Directories
import os, glob, yaml

print("🔍 Unpacking dataset...")
!unzip -q -o /content/gujarat_cctv_dataset.zip -d /content/dataset

train_imgs = glob.glob("/content/**/images/train/*.*", recursive=True)
if len(train_imgs) == 0:
    print("❌ Upload gujarat_cctv_dataset.zip into Colab Files sidebar on the left!")
else:
    train_dir = os.path.dirname(train_imgs[0])
    base_root = os.path.abspath(os.path.join(train_dir, "..", ".."))
    print(f"✅ Found {len(train_imgs)} training images at: {base_root}")
    
    config = {
        "path": base_root,
        "train": "images/train",
        "val": "images/val",
        "names": {
            0: "auto_rickshaw",
            1: "motorcycle",
            2: "scooter",
            3: "car",
            4: "ambulance",
            5: "truck",
            6: "bus",
            7: "van"
        }
    }
    with open("/content/data.yaml", "w") as f:
        yaml.dump(config, f, default_flow_style=False)
    print("✅ Created /content/data.yaml")

In [ ]:
# Cell 4: 1-Click All-In-One Setup & Training on NVIDIA GPU
import os, glob, yaml, shutil, zipfile, torch
from ultralytics import YOLO

# 1. Hardware Check
if not torch.cuda.is_available():
    raise RuntimeError('⚠️ GPU is not active! In Colab menu, click Runtime -> Change runtime type -> select T4 GPU -> Save.')

gpu_name = torch.cuda.get_device_name(0)
gpu_vram = torch.cuda.get_device_properties(0).total_memory / (1024**3)
print(f'⚡ High-Speed NVIDIA GPU Ready: {gpu_name} ({gpu_vram:.1f} GB VRAM)')

# 2. Extract Dataset directly via Python zipfile
zip_path = '/content/gujarat_cctv_dataset.zip'
if not os.path.exists(zip_path):
    for f in os.listdir('/content'):
        if f.endswith('.zip'):
            zip_path = os.path.join('/content', f)
            break

if not os.path.exists(zip_path):
    raise FileNotFoundError('❌ Please upload gujarat_cctv_dataset.zip into the Colab Files sidebar on the left!')

print(f'📦 Unpacking {zip_path}...')
with zipfile.ZipFile(zip_path, 'r') as z:
    z.extractall('/content/dataset')

base_root = '/content/dataset/indian_traffic'
config = {
    'path': base_root,
    'train': 'images/train',
    'val': 'images/val',
    'names': {
        0: 'auto_rickshaw',
        1: 'motorcycle',
        2: 'scooter',
        3: 'car',
        4: 'ambulance',
        5: 'truck',
        6: 'bus',
        7: 'van'
    }
}
with open('/content/data.yaml', 'w') as f:
    yaml.dump(config, f, default_flow_style=False)
print('✅ Dataset verified and /content/data.yaml created!')

# 3. Launch Specialized Training
model = YOLO('yolo12s.pt')
print('🚀 Starting Specialized Two-Wheeler & Small-Car High-Res Training...')

results = model.train(
    data='/content/data.yaml',
    epochs=80,
    imgsz=1024,
    batch=16,
    workers=2,
    device=0,
    optimizer='AdamW',
    lr0=0.0015,
    lrf=0.01,
    weight_decay=0.001,
    warmup_epochs=4,
    cos_lr=True,
    box=8.5,
    cls=1.5,
    dfl=1.8,
    mosaic=1.0,
    mixup=0.20,
    scale=0.75,
    degrees=10.0,
    hsv_h=0.02,
    hsv_s=0.7,
    hsv_v=0.4,
    fliplr=0.5,
    project='/content/sentinel_small_vehicle_training',
    name='two_wheeler_small_car_specialized',
    exist_ok=True,
    verbose=True
)
print('🎉 Training Complete!')


In [ ]:
# Cell 5: Auto-Evaluate Precision on Two-Wheelers & Small Cars
metrics = model.val(imgsz=1280)
print(f"Overall mAP@50: {metrics.box.map50:.4f}")

names = {0: "auto_rickshaw", 1: "motorcycle", 2: "scooter", 3: "car", 4: "ambulance", 5: "truck", 6: "bus", 7: "van"}
for idx, cname in names.items():
    try:
        p = metrics.box.p[idx]
        r = metrics.box.r[idx]
        map50 = metrics.box.maps[idx]
        print(f"  🛵 {cname:15s} -> Precision: {p:.3f} | Recall: {r:.3f} | mAP@50: {map50:.3f}")
    except Exception:
        pass

In [ ]:
# Cell 6: Auto-Download Specialized Model Weights
import shutil
from google.colab import files

best_weights = "/content/sentinel_small_vehicle_training/two_wheeler_small_car_specialized/weights/best.pt"
target_name = "indian_traffic_yolo12_twowheeler_best.pt"

if os.path.exists(best_weights):
    shutil.copy(best_weights, target_name)
    print(f"⬇️ Triggering download of {target_name}...")
    files.download(target_name)
else:
    !find /content -name "best.pt"